# Proyecto final - Turismo, clima y demanda hotelera

Este proyecto analiza la relacion entre turismo, clima, calendario y movilidad para construir un sistema capaz de estimar la demanda hotelera en España. El notebook funcionara como documento de trabajo del proyecto completo: aqui se iran recogiendo la definicion del problema, las fuentes de datos, la arquitectura, las validaciones, el analisis exploratorio, la preparacion del dataset, el modelado y los resultados finales.

## 1. Contexto

El turismo en España tiene una estacionalidad muy marcada. Hay meses donde la demanda sube por vacaciones, festivos, buen tiempo o mayor conectividad aerea, y otros donde cae de forma bastante previsible. Aun asi, esa demanda no depende solo del calendario: tambien influyen el clima, la movilidad y el comportamiento historico de cada provincia.

El proyecto busca construir una base de datos mensual que permita estudiar esa relacion y, mas adelante, entrenar modelos para anticipar la demanda hotelera. La utilidad practica seria ayudar a planificar recursos, detectar picos o caidas de ocupacion y entender mejor que variables explican la demanda turistica en cada zona.

## 2. Problema de Machine Learning

El problema se plantea como **aprendizaje supervisado de regresion**. Es supervisado porque se parte de historico de demanda hotelera ya observada, y es regresion porque el valor a estimar es numerico.

Se busca predecir una variable numerica mensual relacionada con la demanda turistica. El primer objetivo sera trabajar a nivel de **provincia x mes**, porque es un grano suficientemente detallado para captar diferencias territoriales, pero todavia manejable para integrar fuentes heterogeneas.

En resumen:

- Tipo de problema: regresion.
- Tipo de aprendizaje: supervisado.
- Unidad de analisis: provincia y mes.
- Tabla final esperada: una fila por provincia y mes, con variables turisticas, climaticas, calendario y movilidad.

## 3. Variable objetivo

La variable objetivo principal sera:

- `hotel_overnights`: pernoctaciones hoteleras mensuales por provincia.

Tiene sentido usarla como target porque representa volumen real de demanda hotelera. No es solo una tasa o una percepcion indirecta: mide cuantas noches se han consumido en alojamientos hoteleros.

Como alternativa, si durante la integracion se ve que la cobertura de Dataestur es mas estable para otra variable, se podra usar:

- `hotel_occupancy_rate`: porcentaje de ocupacion hotelera mensual.

Ambas son variables continuas, por lo que se podran evaluar con metricas como MAE, RMSE y R2.

## 4. Variables independientes

Las features iniciales salen de cuatro bloques. Si queremos explicar demanda hotelera mensual, necesitamos variables de demanda previa, clima, calendario y movilidad.

| Bloque | Variables candidatas | Justificacion |
|---|---|---|
| Turismo | viajeros, pernoctaciones historicas, ocupacion, estancia media, gasto | Capturan el comportamiento turistico previo y la escala de cada provincia |
| Clima | temperatura media, maxima, minima, precipitacion, horas de lluvia, viento | El tiempo afecta al atractivo de los destinos y puede explicar variaciones mensuales |
| Calendario | mes, temporada, festivos nacionales, festivos regionales | Muchos viajes dependen de vacaciones, puentes y estacionalidad |
| Movilidad | trafico aereo, posibles indicadores ferroviarios o terrestres | Sirve como proxy de conectividad y ayuda a diferenciar turismo internacional, nacional y de proximidad |
| Geografia | provincia, comunidad autonoma | Permite diferenciar patrones territoriales |

A medida que avance el proyecto se podran crear variables derivadas, por ejemplo medias moviles, lags mensuales, indicadores de verano, Semana Santa o periodo COVID.

## 5. Fuentes de datos

Para la primera version integrada del proyecto se priorizan fuentes con relacion clara con el problema y que se puedan defender bien:

- **Dataestur**: fuente principal para turismo. De aqui debe salir el target y parte de las variables turisticas.
- **Open-Meteo**: fuente climatica principal. Es reproducible, no requiere API key y permite descargar historico diario por coordenadas.
- **Festivos**: calendario generado localmente con el paquete `holidays`. Aporta festivos nacionales y regionales.
- **AENA**: datos mensuales de pasajeros, operaciones y carga por aeropuerto. La descarga es manual, pero el procesamiento esta automatizado.
- **Movilidad terrestre**: fuente candidata para capturar mejor turismo nacional y de proximidad. Puede ser especialmente util en provincias donde el aeropuerto no explica bien la llegada de visitantes.

AEMET queda como fuente opcional de contraste. Es oficial, pero para el flujo por defecto se prefiere Open-Meteo porque simplifica la reproduccion del proyecto.

La movilidad terrestre no se fuerza desde el principio para no cargar el proyecto con fuentes poco maduras, pero si encaja con el objetivo: explicar no solo turismo internacional o insular, sino tambien viajes nacionales, escapadas de fin de semana y desplazamientos de proximidad.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "datasets" / "raw"
SILVER = ROOT / "datasets" / "processed" / "silver"
GOLD = ROOT / "datasets" / "processed" / "gold"

print("Proyecto:", ROOT)
print("Raw:", RAW)
print("Silver:", SILVER)
print("Gold:", GOLD)

In [ ]:
sources = []

for source_dir in sorted(RAW.glob("*")):
    if source_dir.is_dir():
        files = [p for p in source_dir.rglob("*") if p.is_file()]
        sources.append({
            "source": source_dir.name,
            "files": len(files),
            "examples": ", ".join(p.name for p in files[:3])
        })

pd.DataFrame(sources)

## 6. Integracion de fuentes

Las fuentes no llegan con el mismo formato:

- Open-Meteo viene como JSON diario por provincia/coordenada.
- Festivos se genera como calendario por fecha y ambito.
- AENA llega en Excel mensual por aeropuerto.
- La movilidad terrestre, si se incorpora, tendra que agregarse tambien a mes y provincia.
- Dataestur se descarga desde API y puede variar segun endpoint.

"Por eso el punto común será `year_month` y la provincia. Las fuentes diarias se agregan a mes. AENA se pasa de aeropuerto a provincia; Dataestur se normaliza para encajar en el mismo grano. Si se añade movilidad terrestre, deberá entrar con la misma lógica para que sea comparable con el resto del dataset."

In [ ]:
silver_files = sorted(SILVER.glob("*.csv"))
pd.DataFrame({
    "file": [p.name for p in silver_files],
    "size_mb": [round(p.stat().st_size / 1024 / 1024, 3) for p in silver_files]
})

In [ ]:
gold_path = GOLD / "tourism_weather_monthly_features.csv"

if gold_path.exists():
    gold = pd.read_csv(gold_path)
    display(gold.head())
    print("Filas:", len(gold))
    print("Columnas:", len(gold.columns))
    print("Rango temporal:", gold["year_month"].min(), "->", gold["year_month"].max())
else:
    print("Todavia no existe la tabla gold integrada.")

In [ ]:
if gold_path.exists():
    candidate_columns = [
        "year_month",
        "region_name",
        "hotel_overnights",
        "hotel_occupancy_rate",
        "temperature_2m_mean",
        "precipitation_sum",
        "holiday_count",
        "aena_passengers",
        "aena_operations",
    ]
    pd.DataFrame({
        "column": candidate_columns,
        "present": [col in gold.columns for col in candidate_columns]
    })

## 7. Arquitectura de datos

La arquitectura se apoya en un data lake con capas:

- `raw` / `bronze`: datos originales tal como llegan o se generan.
- `silver`: datos limpios y normalizados por fuente.
- `gold`: tabla integrada para analisis y modelos.

En local se conserva la misma idea dentro de `datasets/`. En AWS, el repositorio principal sera S3. Esta eleccion encaja bien porque mezcla formatos distintos: JSON, CSV, Parquet y Excel originales. Ademas permite catalogar despues con Glue y consultar con Athena sin mover todos los datos a una base relacional desde el primer dia.

RDS queda como destino para tablas estructuradas o resultados agregados. DocumentDB se reserva para payloads JSON si hace falta trazabilidad de APIs. Lambda y MSK/Kafka forman parte de la arquitectura prevista para automatizacion y componentes de tiempo real, aunque el flujo local no depende de ellos para poder ejecutarse.

## 8. Modelos previstos

Para empezar, se utilizarán los siguientes modelos:

- `LinearRegression` o `Ridge`
- `RandomForestRegressor`
- `XGBoostRegressor`

La evaluacion deberia hacerse respetando el componente temporal. Es decir, evitar mezclar meses futuros en entrenamiento si luego queremos simular una prediccion real.